<a href="https://colab.research.google.com/github/amkathum/DRa/blob/main/DeepLearning4thDay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import os
import imageio as iio
import matplotlib.pyplot as plt
import torch.optim as optim
import numpy as np
import pandas as pd

# Linear Model

In [ ]:
!wget -O Student_Performance.zip https://www.dropbox.com/scl/fi/1lceds9y54dbjoc5eiu0u/Student_Performance.zip?rlkey=ycuxtpgeufmadx9qk4mp9e9z7&dl=0


--2023-11-16 19:40:34--  https://www.dropbox.com/scl/fi/1lceds9y54dbjoc5eiu0u/Student_Performance.zip?rlkey=ycuxtpgeufmadx9qk4mp9e9z7
Resolving www.dropbox.com (www.dropbox.com)... 162.125.65.18, 2620:100:6021:18::a27d:4112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.65.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc18e9d8a1bc735fde5769cfae6b.dl.dropboxusercontent.com/cd/0/inline/CHpDGJtr1AVg6dA84sOG3eA5haRXAmdjodFTq3QgtBOiphK2LpFvXU5OHiHrEjiHD2ljIiuZ1-8NyS5e2eyoJWRB-YEYW2h9zMbo5ELI8z2AxkU_Y9fFdTRsmJm63IO_WDwKTuLAw23KzBl0C8ULCZE_/file# [following]
--2023-11-16 19:40:35--  https://uc18e9d8a1bc735fde5769cfae6b.dl.dropboxusercontent.com/cd/0/inline/CHpDGJtr1AVg6dA84sOG3eA5haRXAmdjodFTq3QgtBOiphK2LpFvXU5OHiHrEjiHD2ljIiuZ1-8NyS5e2eyoJWRB-YEYW2h9zMbo5ELI8z2AxkU_Y9fFdTRsmJm63IO_WDwKTuLAw23KzBl0C8ULCZE_/file
Resolving uc18e9d8a1bc735fde5769cfae6b.dl.dropboxusercontent.com (uc18e9d8a1bc735fde5769cfae6b.dl.dropboxusercontent.com)... 162.12

In [ ]:
!unzip Student_Performance.zip


Archive:  Student_Performance.zip
  inflating: Student_Performance.csv  
  inflating: __MACOSX/._Student_Performance.csv  


In [ ]:
df = pd.read_csv("/content/Student_Performance.csv")
df['Extracurricular Activities'] = df['Extracurricular Activities'].astype('category').cat.codes
X = df.drop("Performance Index",axis = 1)
y = df["Performance Index"]

# Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2)

## Create Tensor Dataset

In [ ]:
# pandas => numpy => tensor
# Tensor is a matrix, pytorch must use tensors
X_train_tensor = torch.tensor(X_train.to_numpy().astype(np.float32))
y_train_tensor = torch.tensor(y_train.to_numpy().astype(np.float32))

train_set = torch.utils.data.TensorDataset(X_train_tensor,y_train_tensor)

In [ ]:
train_set[:][1]

tensor([43., 71., 48.,  ..., 43., 76., 40.])

In [ ]:
X_test_tensor = torch.tensor(X_test.to_numpy().astype(np.float32))
y_test_tensor = torch.tensor(y_test.to_numpy().astype(np.float32))
test_set = torch.utils.data.TensorDataset(X_test_tensor,y_test_tensor)


## Linear Model

In [ ]:
model = nn.Sequential(
    # like a pipeline
    nn.Linear(X.shape[1],39), # linear layer (input -> hidden )
    nn.ReLU(),
    nn.Linear(39,1) # hidden -> output, we chose 1 because we are predicting
    # real value
    # if it was cat or dog => 2
)

## Training Loop

In [ ]:
# Loss function
loss_fn = nn.MSELoss()
# Optimizer - Gradient descent
# w = w - lr * gradient
optimizer = optim.SGD(model.parameters(),lr = 1e-4) # 1e-4 is 0.0001

epoch = 50

for epoch in range(epoch):
  # zero the gradients
  optimizer.zero_grad()
  # from train_set
  X = train_set[:][0] # train_set[row or sample][X or Y]
  y = train_set[:][1]

  prediction = model(X)
  loss = loss_fn(prediction,y)
  loss.backward() # compute gradients
  optimizer.step() # update the weights -lr * gradient

  print(f"Epoch {epoch+1} loss: {loss.item()}")


Epoch1 loss: 836.6624145507812
Epoch2 loss: 574.37158203125
Epoch3 loss: 558.5802001953125
Epoch4 loss: 546.5635986328125
Epoch5 loss: 542.9684448242188
Epoch6 loss: 540.8058471679688
Epoch7 loss: 539.6727905273438
Epoch8 loss: 538.841796875
Epoch9 loss: 538.16650390625
Epoch10 loss: 537.5505981445312
Epoch11 loss: 536.964111328125
Epoch12 loss: 536.3935546875
Epoch13 loss: 535.832275390625
Epoch14 loss: 535.27783203125
Epoch15 loss: 534.7284545898438
Epoch16 loss: 534.1842651367188
Epoch17 loss: 533.6451416015625
Epoch18 loss: 533.110595703125
Epoch19 loss: 532.580322265625
Epoch20 loss: 532.05419921875
Epoch21 loss: 531.5318603515625
Epoch22 loss: 531.0125732421875
Epoch23 loss: 530.496826171875
Epoch24 loss: 529.9844970703125
Epoch25 loss: 529.4755249023438
Epoch26 loss: 528.9697875976562
Epoch27 loss: 528.4666748046875
Epoch28 loss: 527.9663696289062
Epoch29 loss: 527.468505859375
Epoch30 loss: 526.9730224609375
Epoch31 loss: 526.4800415039062
Epoch32 loss: 525.9893798828125
Epoch3

## RMS and Assess The Model

In [ ]:
def rms(y,y_hat):
  return np.sqrt(np.mean(np.square(y-y_hat)))


print(rms(test_set[:][1].detach().numpy(), model(test_set[:][0]).detach().numpy()))
rms(train_set[:][1].detach().numpy(), model(train_set[:][0]).detach().numpy())

22.943188


22.738514

# Classification

In [ ]:

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [ ]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=30)

In [ ]:
type(X_train)

numpy.ndarray

## Tensor Dataset

In [ ]:
X_train_tensor = torch.tensor(X_train.astype(np.float32)) # from double to float
y_train_tensor = torch.tensor(y_train) # int keeep it int
train_set = torch.utils.data.TensorDataset(X_train_tensor,y_train_tensor)

In [ ]:
X_test_tensor = torch.tensor(X_test.astype(np.float32)) # from double to float
y_test_tensor = torch.tensor(y_test) # int keeep it int
test_set = torch.utils.data.TensorDataset(X_test_tensor,y_test_tensor)

## Model

In [ ]:
model = nn.Sequential(
    nn.Linear(X.shape[1],10),
    nn.ReLU(),
    nn.Linear(10,2), # 2 classes
    nn.Softmax(dim=1) # will make it probabilites of classes
    # [0.7,0.3] summation is 1
    # torch.argmax(prediction)
)

## Training Loop

In [ ]:
# Loss function
loss_fn = nn.CrossEntropyLoss()
# Optimizer - Gradient descent
# w = w - lr * gradient
optimizer = optim.Adam(model.parameters(),lr = 1e-4) # 1e-4 is 0.0001

epoch = 20

for epoch in range(epoch):
  # zero the gradients
  optimizer.zero_grad()
  # from train_set
  X = train_set[:][0] # train_set[row or sample][X or Y]
  y = train_set[:][1]

  prediction = model(X)
  loss = loss_fn(prediction,y)
  loss.backward() # compute gradients
  optimizer.step() # update the weights -lr * gradient

  print(f"Epoch {epoch+1} loss: {loss.item()}")


Epoch 1 loss: 0.6824926733970642
Epoch 2 loss: 0.6824926137924194
Epoch 3 loss: 0.6824926137924194
Epoch 4 loss: 0.6824926137924194
Epoch 5 loss: 0.6824926137924194
Epoch 6 loss: 0.6824926137924194
Epoch 7 loss: 0.6824926137924194
Epoch 8 loss: 0.6824926137924194
Epoch 9 loss: 0.6824926137924194
Epoch 10 loss: 0.6824926137924194
Epoch 11 loss: 0.6824926137924194
Epoch 12 loss: 0.6824925541877747
Epoch 13 loss: 0.6824925541877747
Epoch 14 loss: 0.6824925541877747
Epoch 15 loss: 0.6824924945831299
Epoch 16 loss: 0.6824924945831299
Epoch 17 loss: 0.6824924945831299
Epoch 18 loss: 0.6824924945831299
Epoch 19 loss: 0.6824924945831299
Epoch 20 loss: 0.6824924945831299


##Assess Model

In [ ]:
y_pred = model(test_set[:][0])

In [ ]:
torch.argmax(y_pred, dim=1)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [ ]:
accuracy= accuracy_score(torch.argmax(y_pred, dim=1) , y_test)

In [ ]:
accuracy *100

61.40350877192983